# Theorem 23 — Flow necessity under affine distortion

**Formal source:** [`../23_flow_necessity_under_affine_distortion.md`](../23_flow_necessity_under_affine_distortion.md)

This Notebook is an executable finite witness, not the general proof. Passing it supports implementation consistency only; it does not establish learned-model or real-PHM evidence.

In [ ]:
import math
import itertools
import numpy as np
np.set_printoptions(precision=6, suppress=True)


def four_way(projectors, domain_index, atol=1e-9):
    ps = [np.asarray(p, float) for p in projectors]
    dimension = ps[0].shape[0]
    summed = sum(ps)
    values, vectors = np.linalg.eigh((summed + summed.T) / 2)
    basis = vectors[:, np.isclose(values, len(ps), atol=atol)]
    shared = basis @ basis.T if basis.size else np.zeros((dimension, dimension))
    basis = vectors[:, values > atol]
    union = basis @ basis.T if basis.size else np.zeros((dimension, dimension))
    observed = ps[domain_index]
    blocks = [shared, observed - shared, union - observed, np.eye(dimension) - union]
    for projector in blocks:
        np.testing.assert_allclose(projector, projector.T, atol=1e-8)
        np.testing.assert_allclose(projector @ projector, projector, atol=1e-8)
    for index, left in enumerate(blocks):
        for right in blocks[index + 1:]:
            np.testing.assert_allclose(left @ right, 0, atol=1e-8)
    np.testing.assert_allclose(sum(blocks), np.eye(dimension), atol=1e-8)
    return blocks


def normal_pdf(x, mean, standard_deviation):
    return np.exp(-0.5 * ((x - mean) / standard_deviation) ** 2) / (
        math.sqrt(2 * math.pi) * standard_deviation
    )

In [ ]:
rng = np.random.default_rng(23)
anchor = rng.uniform(-1, 1, 30000)
scale, offset = 1.7, -0.4
domain = scale * anchor + offset
recovered = (domain - offset) / scale
affine_error = np.mean((recovered - anchor) ** 2)
assert affine_error < 1e-28
cube = anchor ** 3
design = np.column_stack([cube, np.ones_like(cube)])
parameters = np.linalg.lstsq(design, anchor, rcond=None)[0]
linear_error = np.mean((design @ parameters - anchor) ** 2)
nonlinear_error = np.mean((np.cbrt(cube) - anchor) ** 2)
assert linear_error > 1e-3 and nonlinear_error < 1e-28
print({"affine_inverse_mse": float(affine_error), "affine_on_cube_mse": float(linear_error), "cube_root_mse": float(nonlinear_error)})

In [ ]:
print('THEORY_DEMO_PASS::23_flow_necessity_under_affine_distortion')
print('evidence_level: constructive_or_numerical_witness')
print('formal_claim_supported: false')